# BDAPP — Fine-tune CNN with raised BENIGN class weight (Google Colab, GPU)

This notebook fine-tunes the already-trained flow-to-image CNN (epoch-28 checkpoint from
`BDAPP_train_cnn_colab.ipynb` / `scripts/8_train_cnn.py`), mirroring
`scripts/11_finetune_cnn_benign_weight.py` exactly, with **one targeted change**: BENIGN's
class weight is raised before continuing training.

**Why.** The Notion Work log's Section 15 found BENIGN's balanced class weight (0.179) was a
clear outlier — 2.5x lower than the next-lowest class (DoS Hulk, 0.454) — making it
artificially "cheap" for the weighted loss to misclassify BENIGN. Section 19's Grad-CAM
results confirmed real timing-feature overlap between some BENIGN flows and Bot/Web
Attack/PortScan, so this fine-tune is expected to **reduce**, not fully eliminate, that
leakage — whatever precision gap remains gets documented as a feature-overlap limitation.

**This is NOT a retrain from scratch.** It loads the epoch-28 weights and continues for a
handful of epochs at a 10x lower learning rate (`1e-5` vs. the original `1e-4`), so the
existing convergence isn't undone by a large step.

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)`,
then run cells top to bottom (or `Runtime -> Run all`).

**Outputs go to separate files** — `cnn_resnet18_finetuned_benign_weight.pt`,
`cnn_finetune_*` — so the original epoch-28 checkpoint and its documented baseline numbers
(Section 15) are never overwritten.

## 1. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- go to Runtime > Change runtime type > GPU (T4), then re-run.")

!nvidia-smi


## 2. Mount Google Drive

Same Drive folder the training notebook used (`My Drive/BDAPP_cnn_results/`) — this is
where the epoch-28 checkpoint already lives, and where the fine-tuned checkpoint and
reports will be saved too, under new filenames.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/BDAPP_cnn_results"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print("Reading/writing outputs in:", DRIVE_OUTPUT_DIR)


## 3. Confirm the base checkpoint is there

This fine-tune starts from the real epoch-28 checkpoint, not from scratch — it must
already exist in `DRIVE_OUTPUT_DIR` (saved there by the training notebook).

In [ ]:
BASE_CHECKPOINT_PATH = os.path.join(DRIVE_OUTPUT_DIR, "cnn_resnet18_best.pt")

if not os.path.exists(BASE_CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"{BASE_CHECKPOINT_PATH} not found. Run BDAPP_train_cnn_colab.ipynb first -- "
        "this notebook fine-tunes that checkpoint, it doesn't train from scratch."
    )
print("Found base checkpoint:", BASE_CHECKPOINT_PATH)


## 4. Load the encoded data (via Google Drive -- not files.upload())

Same data as training: `flow_images.zip` already uploaded to your Drive at
`My Drive/BDAPP_data/flow_images.zip` (see `BDAPP_train_cnn_colab.ipynb` if you need to
re-upload it).

In [ ]:
import os

DRIVE_ZIP_PATH = "/content/drive/MyDrive/BDAPP_data/flow_images.zip"

if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(
        f"{DRIVE_ZIP_PATH} not found. Upload flow_images.zip to your Google Drive first "
        "(see BDAPP_train_cnn_colab.ipynb), then fix DRIVE_ZIP_PATH to match its path."
    )
print("Found:", DRIVE_ZIP_PATH)


In [ ]:
import zipfile

with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as zf:
    zf.extractall(".")

if os.path.isdir("flow_images"):
    DATA_DIR = "flow_images"
else:
    DATA_DIR = "."

print("Data dir:", DATA_DIR)
print(os.listdir(DATA_DIR))


## 5. FlowImageDataset

Identical to `scripts/flow_image_dataset.py` -- same nearest-neighbor upscale, same
ImageNet normalization.

In [ ]:
import cv2
import numpy as np
from torch.utils.data import Dataset

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


class FlowImageDataset(Dataset):
    """Wraps a (N, grid_size, grid_size) uint8 array of flow-image grids and a
    (N,) array of integer labels. Each sample is upscaled to cnn_input_size x
    cnn_input_size with NEAREST-NEIGHBOR interpolation only and normalized on
    the fly (see the project's Notion Work log, Section 13)."""

    def __init__(self, grids: np.ndarray, labels: np.ndarray, cnn_input_size: int = 224) -> None:
        if len(grids) != len(labels):
            raise ValueError(f"grids and labels must have the same length, got {len(grids)} and {len(labels)}")
        self.grids = grids
        self.labels = labels
        self.cnn_input_size = cnn_input_size

    def __len__(self) -> int:
        return len(self.grids)

    def __getitem__(self, idx: int):
        grid = self.grids[idx]
        upscaled = cv2.resize(grid, (self.cnn_input_size, self.cnn_input_size), interpolation=cv2.INTER_NEAREST)
        img = upscaled.astype(np.float32) / 255.0
        img = np.repeat(img[np.newaxis, :, :], 3, axis=0)
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        label = int(self.labels[idx])
        return torch.from_numpy(img), label


## 6. Hyperparameters

Same as training (`scripts/config.py`'s `CNN_*`), except the fine-tune-specific ones:
10x lower learning rate, fewer max epochs, shorter patience (this is a short targeted
continuation, not a full training run), and `BENIGN_WEIGHT_MULTIPLIER`, the actual fix
being tested.

In [ ]:
CNN_INPUT_SIZE = 224
CNN_VAL_SIZE = 0.10
CNN_BATCH_SIZE = 128
CNN_NUM_WORKERS = 2
CNN_WEIGHT_DECAY = 1e-4
RANDOM_STATE = 42

# Fine-tune-specific (see config.py's CNN_FINETUNE_* comments for the full reasoning)
CNN_FINETUNE_LEARNING_RATE = 1e-5          # 10x lower than the original 1e-4
CNN_FINETUNE_MAX_EPOCHS = 10
CNN_FINETUNE_PATIENCE = 3
BENIGN_CLASS_NAME = "BENIGN"
BENIGN_WEIGHT_MULTIPLIER = 2.5             # 0.179 -> ~0.45, in line with DoS Hulk's 0.454

# The documented baseline (Work log Section 15) for a direct before/after print at the end,
# on exactly the 3 classes Grad-CAM (Section 19) analyzed.
BASELINE_PRECISION_REFERENCE = {"Bot": 0.15, "Web Attack": 0.22, "PortScan": 0.74}

SMOKE_TEST = False  # set True once to sanity-check the pipeline runs, then back to False
if SMOKE_TEST:
    CNN_FINETUNE_MAX_EPOCHS = 1


## 7. Model, loss, training/eval helpers (same logic as `11_finetune_cnn_benign_weight.py`)

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.utils.class_weight import compute_class_weight


def build_model(n_classes: int) -> nn.Module:
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model


def compute_full_class_weights(y: np.ndarray, n_classes: int) -> np.ndarray:
    present_classes = np.unique(y)
    weights_present = compute_class_weight(class_weight="balanced", classes=present_classes, y=y)
    weights = np.ones(n_classes, dtype=np.float32)
    for cls, w in zip(present_classes, weights_present):
        weights[cls] = w
    return weights


def apply_benign_weight_multiplier(class_weights, class_names):
    """Multiplies BENIGN's balanced weight by BENIGN_WEIGHT_MULTIPLIER, leaving
    every other class's weight untouched."""
    weights = class_weights.copy()
    benign_idx = class_names.index(BENIGN_CLASS_NAME)
    before = weights[benign_idx]
    weights[benign_idx] = before * BENIGN_WEIGHT_MULTIPLIER
    print(f"  BENIGN weight: {before:.3f} -> {weights[benign_idx]:.3f} (x{BENIGN_WEIGHT_MULTIPLIER})")
    return weights


def run_epoch(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, n_samples = 0.0, 0
    all_true, all_pred = [], []

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            n_samples += batch_size

            preds = outputs.argmax(dim=1)
            all_true.extend(labels.cpu().numpy().tolist())
            all_pred.extend(preds.cpu().numpy().tolist())

    return total_loss / n_samples, np.array(all_true), np.array(all_pred)


## 8. Load data, stratified train/val split (test stays untouched)

In [ ]:
import joblib
import os
from sklearn.model_selection import train_test_split

metadata = joblib.load(os.path.join(DATA_DIR, "encoding_metadata.joblib"))
label_encoder = metadata["label_encoder"]
class_names = list(label_encoder.classes_)
n_classes = len(class_names)
print(f"{n_classes} classes: {class_names}")

train_grids_full = np.load(os.path.join(DATA_DIR, "train_grids.npy"))
train_labels_full = np.load(os.path.join(DATA_DIR, "train_labels.npy"))
test_grids = np.load(os.path.join(DATA_DIR, "test_grids.npy"))
test_labels = np.load(os.path.join(DATA_DIR, "test_labels.npy"))
print(f"train: {train_grids_full.shape}  test: {test_grids.shape}")

if SMOKE_TEST:
    train_grids_full, train_labels_full = train_grids_full[:5000], train_labels_full[:5000]
    test_grids, test_labels = test_grids[:1000], test_labels[:1000]
    print(f"[SMOKE_TEST] truncated -- train: {train_grids_full.shape}  test: {test_grids.shape}")

class_counts = np.bincount(train_labels_full, minlength=n_classes)
can_stratify = class_counts[class_counts > 0].min() >= 2

train_grids, val_grids, train_labels, val_labels = train_test_split(
    train_grids_full, train_labels_full,
    test_size=CNN_VAL_SIZE, random_state=RANDOM_STATE,
    stratify=train_labels_full if can_stratify else None,
)
print(f"Train: {len(train_grids):,}  Val: {len(val_grids):,}  Test: {len(test_grids):,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

from torch.utils.data import DataLoader

train_ds = FlowImageDataset(train_grids, train_labels, CNN_INPUT_SIZE)
val_ds = FlowImageDataset(val_grids, val_labels, CNN_INPUT_SIZE)
test_ds = FlowImageDataset(test_grids, test_labels, CNN_INPUT_SIZE)

train_loader = DataLoader(train_ds, batch_size=CNN_BATCH_SIZE, shuffle=True,
                           num_workers=CNN_NUM_WORKERS, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=CNN_BATCH_SIZE, shuffle=False,
                         num_workers=CNN_NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_ds, batch_size=CNN_BATCH_SIZE, shuffle=False,
                          num_workers=CNN_NUM_WORKERS, pin_memory=(device.type == "cuda"))


## 9. Load the epoch-28 checkpoint as the fine-tune starting point

In [ ]:
model = build_model(n_classes).to(device)

base_checkpoint = torch.load(BASE_CHECKPOINT_PATH, map_location=device)
model.load_state_dict(base_checkpoint["model_state_dict"])
print(f"Loaded epoch {base_checkpoint['epoch']} (val macro-F1={base_checkpoint['val_macro_f1']:.4f}) "
      "as the fine-tune starting point.")


## 10. Class weights: balanced, then BENIGN boosted

In [ ]:
class_weights = compute_full_class_weights(train_labels, n_classes)
print("Balanced class weights:", dict(zip(class_names, class_weights.round(3))))

class_weights = apply_benign_weight_multiplier(class_weights, class_names)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)


## 11. Fine-tune (early stopping on validation macro-F1, resumable across sessions)

Same resumable-checkpoint pattern as the training notebook: saves to Drive every time val
macro-F1 improves, rewrites the training-log CSV after every epoch. Saved to **separate
filenames** (`cnn_resnet18_finetuned_benign_weight.pt`, `cnn_finetune_*`) so the original
epoch-28 checkpoint is never touched.

In [ ]:
import time
import csv
import pandas as pd
from sklearn.metrics import f1_score

RESUME = True  # set False to force a fresh start from the epoch-28 base checkpoint

FINETUNE_MODEL_PATH = os.path.join(DRIVE_OUTPUT_DIR, "cnn_resnet18_finetuned_benign_weight.pt")
FINETUNE_TRAINING_LOG_CSV = os.path.join(DRIVE_OUTPUT_DIR, "cnn_finetune_training_log.csv")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=CNN_FINETUNE_LEARNING_RATE, weight_decay=CNN_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

start_epoch = 1
best_val_f1 = -1.0
epochs_without_improvement = 0
log_rows = []

if RESUME and os.path.exists(FINETUNE_MODEL_PATH):
    print(f"Found existing fine-tune checkpoint at {FINETUNE_MODEL_PATH} -- resuming instead of restarting.")
    checkpoint = torch.load(FINETUNE_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    best_val_f1 = checkpoint["val_macro_f1"]
    start_epoch = checkpoint["epoch"] + 1

    if os.path.exists(FINETUNE_TRAINING_LOG_CSV):
        prev_log = pd.read_csv(FINETUNE_TRAINING_LOG_CSV)
        log_rows = prev_log.to_dict("records")
        last_logged_epoch = int(prev_log["epoch"].max())
        best_epoch = int(prev_log.loc[prev_log["val_macro_f1"].idxmax(), "epoch"])
        epochs_without_improvement = last_logged_epoch - best_epoch
    print(f"  Resuming from epoch {start_epoch}, best val macro-F1 so far = {best_val_f1:.4f}")
else:
    print(f"Starting fine-tune fresh from the epoch-{base_checkpoint['epoch']} base checkpoint.")

print(f"Fine-tuning up to epoch {CNN_FINETUNE_MAX_EPOCHS} "
      f"(lr={CNN_FINETUNE_LEARNING_RATE:.1e}, patience={CNN_FINETUNE_PATIENCE}, metric=val macro-F1)...\n")
for epoch in range(start_epoch, CNN_FINETUNE_MAX_EPOCHS + 1):
    start = time.time()

    train_loss, _, _ = run_epoch(model, train_loader, criterion, device, optimizer)
    val_loss, val_true, val_pred = run_epoch(model, val_loader, criterion, device, optimizer=None)

    val_accuracy = float((val_true == val_pred).mean())
    val_macro_f1 = f1_score(val_true, val_pred, average="macro", zero_division=0)
    scheduler.step(val_macro_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    elapsed = time.time() - start
    print(f"Epoch {epoch:3d}/{CNN_FINETUNE_MAX_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
          f"val_acc={val_accuracy:.4f} | val_macro_f1={val_macro_f1:.4f} | lr={current_lr:.2e} | {elapsed:.1f}s")

    log_rows.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                      "val_accuracy": val_accuracy, "val_macro_f1": val_macro_f1, "lr": current_lr})

    with open(FINETUNE_TRAINING_LOG_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(log_rows[0].keys()))
        writer.writeheader()
        writer.writerows(log_rows)

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        epochs_without_improvement = 0
        torch.save({"model_state_dict": model.state_dict(),
                     "epoch": epoch, "val_macro_f1": val_macro_f1,
                     "class_names": class_names,
                     "base_checkpoint_epoch": base_checkpoint["epoch"],
                     "benign_weight_multiplier": BENIGN_WEIGHT_MULTIPLIER}, FINETUNE_MODEL_PATH)
        print(f"  -> New best val macro-F1 ({val_macro_f1:.4f}). Saved.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CNN_FINETUNE_PATIENCE:
            print(f"\nEarly stopping: no improvement for {CNN_FINETUNE_PATIENCE} epochs "
                  f"(best={best_val_f1:.4f} at epoch {epoch - epochs_without_improvement}).")
            break

print("Fine-tune training log saved to", FINETUNE_TRAINING_LOG_CSV)


## 12. Fine-tune training curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = [r["epoch"] for r in log_rows]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(epochs_ran, [r["train_loss"] for r in log_rows], label="train")
axes[0].plot(epochs_ran, [r["val_loss"] for r in log_rows], label="val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Fine-tune loss per epoch"); axes[0].legend()

axes[1].plot(epochs_ran, [r["val_accuracy"] for r in log_rows], label="val accuracy")
axes[1].plot(epochs_ran, [r["val_macro_f1"] for r in log_rows], label="val macro-F1")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score"); axes[1].set_title("Fine-tune validation metrics per epoch"); axes[1].legend()

plt.tight_layout()
FINETUNE_CURVES_PNG = os.path.join(DRIVE_OUTPUT_DIR, "cnn_finetune_training_curves.png")
fig.savefig(FINETUNE_CURVES_PNG, dpi=150)
plt.show()
print("Saved to", FINETUNE_CURVES_PNG)


## 13. Final evaluation on the untouched test set (best fine-tuned checkpoint)

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

checkpoint = torch.load(FINETUNE_MODEL_PATH, map_location=device)
print(f"Loading best fine-tuned checkpoint (epoch {checkpoint['epoch']}, val macro-F1={checkpoint['val_macro_f1']:.4f})")
model.load_state_dict(checkpoint["model_state_dict"])

_, test_true, test_pred = run_epoch(model, test_loader, criterion, device, optimizer=None)

all_labels = np.arange(n_classes)
report_dict = classification_report(test_true, test_pred, labels=all_labels, target_names=class_names,
                                     output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).T
FINETUNE_REPORT_CSV = os.path.join(DRIVE_OUTPUT_DIR, "cnn_finetune_classification_report.csv")
report_df.to_csv(FINETUNE_REPORT_CSV)

print(classification_report(test_true, test_pred, labels=all_labels, target_names=class_names, zero_division=0))

cm = confusion_matrix(test_true, test_pred, labels=all_labels)
fig, ax = plt.subplots(figsize=(10, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", values_format="d", colorbar=False)
ax.set_title("CNN (ResNet18) - Confusion Matrix, fine-tuned (test set)")
plt.tight_layout()
FINETUNE_CM_PNG = os.path.join(DRIVE_OUTPUT_DIR, "cnn_finetune_confusion_matrix.png")
fig.savefig(FINETUNE_CM_PNG, dpi=150)
plt.show()
print("Saved report and confusion matrix.")


## 14. Before / after: exactly the 3 classes Grad-CAM analyzed (Work log Section 19)

In [ ]:
print("PRECISION: BASELINE (Section 15) vs FINE-TUNED (this run)")
print("=" * 60)
for cls_name, baseline_precision in BASELINE_PRECISION_REFERENCE.items():
    new_precision = report_dict.get(cls_name, {}).get("precision")
    if new_precision is None:
        continue
    delta = new_precision - baseline_precision
    arrow = "UP" if delta > 0 else ("DOWN" if delta < 0 else "=")
    print(f"  {cls_name:<12s}  {baseline_precision:.2f} -> {new_precision:.2f}  ({arrow} {delta:+.2f})")


## 15. Where your results are

Everything was saved to your Google Drive at `My Drive/BDAPP_cnn_results/` as training
progressed — `cnn_resnet18_finetuned_benign_weight.pt`, `cnn_finetune_training_log.csv`,
`cnn_finetune_classification_report.csv`, `cnn_finetune_training_curves.png`,
`cnn_finetune_confusion_matrix.png` — alongside (not overwriting) the original epoch-28
files from the training notebook.

Put the files back into your local project at:

- `cnn_resnet18_finetuned_benign_weight.pt` -> `models/`
- `cnn_finetune_*.csv`, `cnn_finetune_*.png` -> `data/processed/reports/`

Then paste the printed before/after precision table (cell 14 above) and the full
classification report back into chat so the Notion Work log can be updated with the real,
verified fine-tune results.

In [ ]:
print("Saved in", DRIVE_OUTPUT_DIR, ":")
for fname in sorted(os.listdir(DRIVE_OUTPUT_DIR)):
    fpath = os.path.join(DRIVE_OUTPUT_DIR, fname)
    print(f"  {fname}  ({os.path.getsize(fpath) / 1024:.1f} KB)")
